# Practical 6: Reinforcement Learning - Q-Learning in a Maze
**Problem Statement:** Implement Reinforcement Learning using the Q-Learning algorithm in a maze environment. The agent must learn to navigate from a start position to a goal position while avoiding obstacles. By interacting with the environment and receiving rewards or penalties, the agent learns an optimal path through trial and error.

No dataset is needed: the maze is a small grid built with NumPy.

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import random

np.random.seed(42)
random.seed(42)
%matplotlib inline

## 2. Define the Maze Environment
- `0` = free cell, `1` = obstacle (wall)
- **Start** = top-left, **Goal** = bottom-right
- **Actions:** 0 = Up, 1 = Down, 2 = Left, 3 = Right

**Rewards**
| Event | Reward |
|---|---|
| Reach the goal | +100 |
| Hit a wall or the boundary (agent stays in place) | -10 |
| Any normal move | -1 (encourages the shortest path) |

In [ ]:
maze = np.array([
    [0, 0, 0, 1, 0, 0],
    [1, 1, 0, 1, 0, 1],
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 1, 0],
    [1, 1, 1, 0, 0, 0],
])

START = (0, 0)
GOAL = (5, 5)
ROWS, COLS = maze.shape

ACTIONS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
ACTION_NAMES = {0: "Up", 1: "Down", 2: "Left", 3: "Right"}
ARROWS = {0: "↑", 1: "↓", 2: "←", 3: "→"}

REWARD_GOAL = 100
REWARD_WALL = -10
REWARD_STEP = -1

print("Maze size:", maze.shape)
print("Start:", START, " Goal:", GOAL)

In [ ]:
class MazeEnv:
    def __init__(self, maze, start, goal):
        self.maze = maze
        self.start = start
        self.goal = goal
        self.state = start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        dr, dc = ACTIONS[action]
        r, c = self.state
        nr, nc = r + dr, c + dc

        # Outside the grid or into an obstacle: stay in place and get a penalty
        if nr < 0 or nr >= ROWS or nc < 0 or nc >= COLS or self.maze[nr, nc] == 1:
            return self.state, REWARD_WALL, False

        self.state = (nr, nc)
        if self.state == self.goal:
            return self.state, REWARD_GOAL, True
        return self.state, REWARD_STEP, False

env = MazeEnv(maze, START, GOAL)

## 3. Visualize the Maze

In [ ]:
def draw_maze(path=None, title="Maze"):
    fig, ax = plt.subplots(figsize=(6, 6))
    cmap = ListedColormap(["white", "dimgray"])
    ax.imshow(maze, cmap=cmap)

    ax.text(START[1], START[0], "S", ha="center", va="center", fontsize=18, color="green", fontweight="bold")
    ax.text(GOAL[1], GOAL[0], "G", ha="center", va="center", fontsize=18, color="red", fontweight="bold")

    if path:
        ys = [p[0] for p in path]
        xs = [p[1] for p in path]
        ax.plot(xs, ys, color="royalblue", linewidth=3, marker="o", markersize=6)

    ax.set_xticks(np.arange(-0.5, COLS, 1))
    ax.set_yticks(np.arange(-0.5, ROWS, 1))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(color="black", linewidth=1)
    ax.set_title(title)
    plt.show()

draw_maze(title="Maze Environment (dark = obstacle, S = start, G = goal)")

## 4. Q-Learning Algorithm
The agent keeps a **Q-table** `Q[state, action]`, the expected future reward of taking an action in a state. After each step it updates:

**Q(s, a) ← Q(s, a) + α [ r + γ · max Q(s', a') − Q(s, a) ]**

- α (alpha) = learning rate
- γ (gamma) = discount factor
- **ε-greedy policy:** with probability ε explore (random action), otherwise exploit (best known action). ε decays over time.

In [ ]:
# Hyperparameters
alpha = 0.1            # learning rate
gamma = 0.9            # discount factor
epsilon = 1.0          # start fully exploring
epsilon_min = 0.01
epsilon_decay = 0.995
episodes = 1000
max_steps = 200        # step limit per episode

# Q-table: one value per (row, col, action)
Q = np.zeros((ROWS, COLS, len(ACTIONS)))

rewards_per_episode = []
steps_per_episode = []
epsilon_history = []

## 5. Train the Agent

In [ ]:
for ep in range(episodes):
    state = env.reset()
    total_reward = 0

    for step in range(max_steps):
        # ε-greedy action selection
        if random.random() < epsilon:
            action = random.choice(list(ACTIONS.keys()))
        else:
            action = int(np.argmax(Q[state[0], state[1]]))

        next_state, reward, done = env.step(action)

        # Q-learning update
        best_next = np.max(Q[next_state[0], next_state[1]])
        Q[state[0], state[1], action] += alpha * (reward + gamma * best_next - Q[state[0], state[1], action])

        state = next_state
        total_reward += reward
        if done:
            break

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    rewards_per_episode.append(total_reward)
    steps_per_episode.append(step + 1)
    epsilon_history.append(epsilon)

    if (ep + 1) % 200 == 0:
        print(f"Episode {ep+1:4d} | Reward: {total_reward:7.1f} | Steps: {step+1:3d} | Epsilon: {epsilon:.3f}")

print("\nTraining complete.")

## 6. Learning Curves

In [ ]:
def smooth(data, window=20):
    return np.convolve(data, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].plot(rewards_per_episode, alpha=0.3, label="Raw")
axes[0].plot(range(19, episodes), smooth(rewards_per_episode), color="red", label="Smoothed (20 episodes)")
axes[0].set_title("Total Reward per Episode")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Reward")
axes[0].legend()

axes[1].plot(steps_per_episode, alpha=0.3, label="Raw")
axes[1].plot(range(19, episodes), smooth(steps_per_episode), color="red", label="Smoothed (20 episodes)")
axes[1].set_title("Steps per Episode")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Steps")
axes[1].legend()

axes[2].plot(epsilon_history, color="green")
axes[2].set_title("Epsilon Decay (Exploration Rate)")
axes[2].set_xlabel("Episode")
axes[2].set_ylabel("Epsilon")

plt.tight_layout()
plt.show()

As training goes on, reward rises and the number of steps falls, which shows the agent is learning a shorter route to the goal.

## 7. Learned Policy (best action in every cell)

In [ ]:
policy_grid = np.full((ROWS, COLS), " ", dtype=object)
for r in range(ROWS):
    for c in range(COLS):
        if maze[r, c] == 1:
            policy_grid[r, c] = "■"
        elif (r, c) == GOAL:
            policy_grid[r, c] = "G"
        else:
            policy_grid[r, c] = ARROWS[int(np.argmax(Q[r, c]))]

print("Learned policy (■ = obstacle, G = goal):\n")
for row in policy_grid:
    print("  ".join(row))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.imshow(maze, cmap=ListedColormap(["white", "dimgray"]))
for r in range(ROWS):
    for c in range(COLS):
        if maze[r, c] == 0 and (r, c) != GOAL:
            ax.text(c, r, ARROWS[int(np.argmax(Q[r, c]))], ha="center", va="center", fontsize=20, color="navy")
ax.text(GOAL[1], GOAL[0], "G", ha="center", va="center", fontsize=18, color="red", fontweight="bold")
ax.set_xticks(np.arange(-0.5, COLS, 1))
ax.set_yticks(np.arange(-0.5, ROWS, 1))
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.grid(color="black", linewidth=1)
ax.set_title("Learned Policy")
plt.show()

## 8. Extract the Optimal Path

In [ ]:
def get_optimal_path(max_len=100):
    state = START
    path = [state]
    for _ in range(max_len):
        action = int(np.argmax(Q[state[0], state[1]]))
        dr, dc = ACTIONS[action]
        nr, nc = state[0] + dr, state[1] + dc
        if nr < 0 or nr >= ROWS or nc < 0 or nc >= COLS or maze[nr, nc] == 1:
            break
        state = (nr, nc)
        path.append(state)
        if state == GOAL:
            break
    return path

optimal_path = get_optimal_path()
reached = optimal_path[-1] == GOAL

print("Path found:", optimal_path)
print("Reached goal:", reached)
print("Number of steps:", len(optimal_path) - 1)
draw_maze(path=optimal_path, title="Optimal Path Learned by the Agent")

## 9. Verify with a Pure Greedy Run (no exploration)

In [ ]:
successes = 0
trials = 100
for _ in range(trials):
    state = env.reset()
    for _ in range(max_steps):
        action = int(np.argmax(Q[state[0], state[1]]))
        state, reward, done = env.step(action)
        if done:
            successes += 1
            break

print(f"Greedy agent reached the goal in {successes}/{trials} runs")

## 10. Q-Table Heatmap (best Q-value per cell)

In [ ]:
best_q = np.max(Q, axis=2).astype(float)
best_q[maze == 1] = np.nan

plt.figure(figsize=(7, 6))
plt.imshow(best_q, cmap="viridis")
plt.colorbar(label="max Q-value")
for r in range(ROWS):
    for c in range(COLS):
        if maze[r, c] == 0:
            plt.text(c, r, f"{best_q[r, c]:.0f}", ha="center", va="center", color="white", fontsize=9)
plt.title("Learned State Values (cells near the goal have higher value)")
plt.xticks([])
plt.yticks([])
plt.show()

## Conclusion
- The maze was modelled as a grid environment with rewards: +100 for the goal, -10 for hitting a wall and -1 per step.
- The agent used the **Q-Learning** algorithm with an ε-greedy policy. Early on it explores randomly, and as ε decays it exploits what it has learned.
- The learning curves show reward increasing and steps per episode dropping as training progresses.
- The learned policy and path show the agent finds the shortest route from start to goal while avoiding obstacles.
- The Q-values are highest near the goal and fall off with distance, because of the discount factor γ.